# R1-4 Duplicate-Aware Sensitivity Analysis — 10-Seed Robust Co-occurrence v7

**v7 correction:** Gensim aggregate C_NPMI/UMass is retained only as an audit column. The manuscript sensitivity table uses one deterministic document-level co-occurrence implementation for **all 30 runs**, including support for multi-word/n-gram topic terms. This avoids NaN propagation without mixing metric definitions across runs.

Seed 1–3 quick caches are reused when available; missing seeds 4–10 are trained.


## 0A. Install / verify required dependencies

Cell ini memastikan dependency yang dipakai `topic_metrics.py` tersedia, terutama `gensim` untuk menghitung C_v, C_NPMI, dan UMass.


In [ ]:
# ============================================================
# 0A. INSTALL / VERIFY REQUIRED DEPENDENCIES
# ============================================================

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "gensim": "gensim==4.4.0",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "sklearn": "scikit-learn",
}

to_install = []

for module_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        to_install.append(pip_name)

if to_install:
    print("Installing missing packages:")
    for pkg in to_install:
        print("  -", pkg)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *to_install]
    )
else:
    print("All required packages are already installed.")

# Explicit verification.
import gensim
import pyarrow
import joblib
import sklearn

print("\nDependency versions:")
print("  gensim       :", gensim.__version__)
print("  pyarrow      :", pyarrow.__version__)
print("  joblib       :", joblib.__version__)
print("  scikit-learn :", sklearn.__version__)
print("\n✓ Dependencies ready.")


## 0. Google Colab / Project Setup

Jalankan cell ini **sebelum semua import HMRNet**. Cell akan mount Google Drive, berpindah ke root project, menambahkan project ke `sys.path`, dan memeriksa folder `src`.


In [ ]:
# ============================================================
# 0. ROBUST GOOGLE DRIVE SETUP + LOCAL CACHE
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
DRIVE_ROOT = Path("/content/drive")
FEATURE_DIR_DRIVE = PROJECT_ROOT / "results/checkpoints/features"
FEATURE_DIR_LOCAL = Path("/content/hmrnet_features_cache")

def drive_is_healthy() -> bool:
    try:
        test_path = Path("/content/drive/MyDrive")
        if not test_path.exists():
            return False
        # Force an actual directory read, not only exists().
        _ = next(test_path.iterdir(), None)
        return True
    except (OSError, IOError):
        return False

def repair_google_drive():
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Google Colab.")
        return

    if drive_is_healthy():
        print("Google Drive mount is healthy.")
        return

    print("Google Drive endpoint is unavailable. Repairing mount...")

    # Best-effort unmount of stale FUSE endpoint.
    try:
        drive.flush_and_unmount()
    except Exception:
        pass

    try:
        subprocess.run(
            ["fusermount", "-u", "/content/drive"],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except Exception:
        pass

    try:
        if DRIVE_ROOT.exists():
            shutil.rmtree(DRIVE_ROOT, ignore_errors=True)
    except Exception:
        pass

    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    # Force a fresh mount.
    drive.mount("/content/drive", force_remount=True)

    if not drive_is_healthy():
        raise RuntimeError(
            "Google Drive masih tidak dapat diakses setelah remount. "
            "Gunakan Runtime > Restart session, lalu jalankan cell ini kembali."
        )

repair_google_drive()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root tidak ditemukan:\n{PROJECT_ROOT}\n\n"
        "Periksa lokasi folder project di Google Drive."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_DIR = PROJECT_ROOT / "src"
required_src = [
    SRC_DIR / "preprocessing.py",
    SRC_DIR / "hmrnet.py",
    SRC_DIR / "topic_metrics.py",
    SRC_DIR / "stability_metrics.py",
]
missing_src = [str(p) for p in required_src if not p.exists()]
if missing_src:
    raise FileNotFoundError(
        "Source code HMRNet tidak lengkap:\n" + "\n".join(missing_src)
    )

# ------------------------------------------------------------
# Copy feature bundle from Drive -> local Colab storage.
# PyArrow/parquet is more reliable from local storage than FUSE Drive.
# ------------------------------------------------------------
required_feature_files = [
    "texts.parquet",
]

if not FEATURE_DIR_DRIVE.exists():
    raise FileNotFoundError(
        f"Feature directory tidak ditemukan:\n{FEATURE_DIR_DRIVE}"
    )

for name in required_feature_files:
    if not (FEATURE_DIR_DRIVE / name).exists():
        raise FileNotFoundError(
            f"Feature bundle tidak lengkap. File hilang: "
            f"{FEATURE_DIR_DRIVE / name}"
        )

if FEATURE_DIR_LOCAL.exists():
    shutil.rmtree(FEATURE_DIR_LOCAL)

print("\nCopying fixed feature bundle to local Colab storage...")
shutil.copytree(FEATURE_DIR_DRIVE, FEATURE_DIR_LOCAL)

# Verify local Parquet is readable before proceeding.
import pandas as pd
probe = pd.read_parquet(FEATURE_DIR_LOCAL / "texts.parquet")
print(f"✓ Local texts.parquet readable: {len(probe):,} rows")
del probe

# IMPORTANT:
# Downstream notebook cells should use this local directory.
FEATURE_DIR = FEATURE_DIR_LOCAL

print("\nPROJECT_ROOT       :", PROJECT_ROOT)
print("SRC_DIR            :", SRC_DIR)
print("FEATURE_DIR_DRIVE  :", FEATURE_DIR_DRIVE)
print("FEATURE_DIR (local):", FEATURE_DIR)
print("CURRENT DIR        :", Path.cwd())
print("\n✓ Drive repaired, project import path configured, and feature bundle cached locally.")


### Jika masih muncul `Transport endpoint is not connected`

1. Pilih **Runtime → Restart session**.
2. Jalankan kembali notebook **dari cell 0**.
3. Jangan menjalankan cell `load_feature_bundle()` sebelum cell setup selesai.

Notebook ini membaca `texts.parquet` dari **cache lokal `/content/hmrnet_features_cache`**, bukan langsung dari Google Drive, untuk menghindari error FUSE/PyArrow saat proses eksperimen.


In [ ]:
# -*- coding: utf-8 -*-
"""R1-4 Duplicate-Aware Sensitivity Analysis for HMRNet-Final.
Run in Google Colab from the existing HMRNet project.
"""
from __future__ import annotations
import copy, gc, html, json, random, re, unicodedata, warnings
from collections import defaultdict
from dataclasses import asdict
from difflib import SequenceMatcher
from pathlib import Path
import joblib, numpy as np, pandas as pd, torch
from scipy.spatial.distance import jensenshannon
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.neighbors import NearestNeighbors
from src.preprocessing import load_feature_bundle
from src.hmrnet import HMRNet, HMRNetConfig
from src.topic_metrics import evaluate_topics, compute_coherence_scores
from src.stability_metrics import evaluate_stability_runs
warnings.filterwarnings('ignore')


## Paths


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/UDB/Riset/2026')
# FEATURE_DIR sudah dibuat oleh cell setup sebagai local cache.
FEATURE_DIR = globals().get('FEATURE_DIR', PROJECT_ROOT/'results/checkpoints/features')
FINAL_CHECKPOINT = PROJECT_ROOT/'results/checkpoints/HMRNet-Stage8-v2-CV-GT070.pt'
OUTPUT_DIR = PROJECT_ROOT/'results/duplicate_sensitivity'
CORPUS_DIR = OUTPUT_DIR/'corpora'
MODEL_DIR = OUTPUT_DIR/'checkpoints'
RUN_DIR = OUTPUT_DIR/'runs'
TABLE_DIR = OUTPUT_DIR/'tables'
for p in [OUTPUT_DIR, CORPUS_DIR, MODEL_DIR, RUN_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('FEATURE_DIR used by experiment:', FEATURE_DIR)


## Run mode


In [ ]:
MODE='manuscript'
SEEDS=list(range(1,11))
FORCE_RETRAIN=False
TOP_N_WORDS=10
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('MODE:', MODE, '| SEEDS:', SEEDS, '| DEVICE:', DEVICE)


## Duplicate criteria


In [ ]:
TFIDF_COSINE_CANDIDATE = 0.85
TOKEN_JACCARD_THRESHOLD = 0.90
SEQUENCE_RATIO_THRESHOLD = 0.93
K_NEIGHBORS = 12

def normalize_text(value):
    s = '' if pd.isna(value) else str(value)
    s = html.unescape(s)
    s = unicodedata.normalize('NFKC', s).lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'#(\w+)', r'\1', s)
    s = re.sub(r'[^\w\s]', ' ', s, flags=re.UNICODE)
    s = re.sub(r'_+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def token_jaccard(a, b):
    A, B = set(a.split()), set(b.split())
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

class UF:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0] * n

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return
        if self.r[a] < self.r[b]:
            a, b = b, a
        self.p[b] = a
        if self.r[a] == self.r[b]:
            self.r[a] += 1

print("Duplicate criteria ready.")


## Fixed feature bundle → duplicate-aware sensitivity indices

Deduplication dilakukan langsung pada teks di feature bundle yang menghasilkan BoW, semantic embedding, dan document embedding. Karena itu indeks pasti konsisten dengan fitur HMRNet.


In [ ]:
# ============================================================
# LOAD FIXED FEATURE BUNDLE
# ============================================================

b = load_feature_bundle(FEATURE_DIR)

raw = list(b['raw_texts'])
processed = list(b['processed_texts'])
bow = b['bow']
sem = b['semantic_embeddings']
doc = b['document_embeddings']
vocab = b['vocabulary']

n_docs = len(raw)

checks = {
    'raw_texts': len(raw),
    'processed_texts': len(processed),
    'bow_rows': int(bow.shape[0]),
    'semantic_rows': int(sem.shape[0]),
    'document_rows': int(doc.shape[0]),
}

print("Feature-bundle row counts:")
for key, value in checks.items():
    print(f"  {key:18s}: {value:,}")

if len(set(checks.values())) != 1:
    raise ValueError(f"Internal feature-bundle row mismatch: {checks}")

if n_docs != 23370:
    raise ValueError(
        f"Expected manuscript corpus size 23,370, but bundle has {n_docs:,} rows."
    )

print("\n✓ Fixed feature bundle internally aligned.")
print("✓ Manuscript corpus size = 23,370.")

# ============================================================
# BUILD EXACT-NORMALIZED CLUSTERS FROM FEATURE-BUNDLE TEXTS
# ============================================================

feature_df = pd.DataFrame({
    'feature_index': np.arange(n_docs, dtype=int),
    'raw_text': ['' if x is None else str(x) for x in raw],
    'processed_text': ['' if x is None else str(x) for x in processed],
})

feature_df['_norm'] = feature_df['raw_text'].map(normalize_text)

# If a bundle raw text is empty, use its processed text only for duplicate screening.
empty_mask = feature_df['_norm'].eq('')
if empty_mask.any():
    feature_df.loc[empty_mask, '_norm'] = (
        feature_df.loc[empty_mask, 'processed_text'].map(normalize_text)
    )

if feature_df['_norm'].eq('').any():
    bad = feature_df.loc[feature_df['_norm'].eq(''), 'feature_index'].tolist()[:20]
    raise ValueError(
        "Empty text remains after fallback to processed_text. "
        f"First feature indices: {bad}"
    )

codes, _ = pd.factorize(feature_df['_norm'], sort=True)
feature_df['exact_cluster_id'] = codes

exact_sizes = feature_df.groupby('exact_cluster_id').size()
feature_df['exact_cluster_size'] = (
    feature_df['exact_cluster_id'].map(exact_sizes).astype(int)
)

# Keep the first row in fixed feature-bundle order.
exact_rep_map = (
    feature_df.groupby('exact_cluster_id')['feature_index']
    .min()
    .astype(int)
    .to_dict()
)

feature_df['exact_rep'] = (
    feature_df['exact_cluster_id'].map(exact_rep_map).astype(int)
)

feature_df['keep_exact'] = (
    feature_df['feature_index'] == feature_df['exact_rep']
)

exact_keep = np.sort(
    feature_df.loc[
        feature_df['keep_exact'],
        'feature_index'
    ].to_numpy(dtype=int)
)

print("\nExact-normalized dedup:")
print("  Retained :", f"{len(exact_keep):,}")
print("  Removed  :", f"{n_docs-len(exact_keep):,}",
      f"({100*(n_docs-len(exact_keep))/n_docs:.2f}%)")

# ============================================================
# CONSERVATIVE NEAR-DUPLICATE CLUSTERING
# ============================================================

rep = (
    feature_df.set_index('feature_index')
    .loc[exact_keep]
    .reset_index()
)
texts = rep['_norm'].tolist()

vec = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=120000,
    norm='l2',
    sublinear_tf=True,
)
X = vec.fit_transform(texts)

k = min(K_NEIGHBORS, len(texts))
nn = NearestNeighbors(
    n_neighbors=k,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1,
).fit(X)

dist, nei = nn.kneighbors(X, return_distance=True)

uf = UF(len(texts))
seen = set()
accepted_pairs = []

for i in range(len(texts)):
    for d, j in zip(dist[i, 1:], nei[i, 1:]):
        cosine_sim = 1.0 - float(d)

        if cosine_sim < TFIDF_COSINE_CANDIDATE:
            continue

        a, bb = sorted((i, int(j)))

        if a == bb or (a, bb) in seen:
            continue

        seen.add((a, bb))

        s1, s2 = texts[a], texts[bb]
        jac = token_jaccard(s1, s2)

        if jac < TOKEN_JACCARD_THRESHOLD:
            continue

        seq = SequenceMatcher(
            None, s1, s2, autojunk=False
        ).ratio()

        if seq < SEQUENCE_RATIO_THRESHOLD:
            continue

        uf.union(a, bb)
        accepted_pairs.append(
            (a, bb, cosine_sim, jac, seq)
        )

components = defaultdict(list)
for i in range(len(texts)):
    components[uf.find(i)].append(i)

ordered_components = sorted(
    components.values(),
    key=lambda members: min(
        int(rep.iloc[p].feature_index)
        for p in members
    ),
)

rep_to_cluster = {}
cluster_rep = {}
cluster_size = {}

for cid, members in enumerate(ordered_components):
    idxs = [
        int(rep.iloc[p].feature_index)
        for p in members
    ]
    keep = min(idxs)

    cluster_rep[cid] = keep
    cluster_size[cid] = len(members)

    for p in members:
        rep_to_cluster[int(rep.iloc[p].feature_index)] = cid

feature_df['near_cid'] = (
    feature_df['exact_rep']
    .map(rep_to_cluster)
    .astype(int)
)
feature_df['near_rep'] = (
    feature_df['near_cid']
    .map(cluster_rep)
    .astype(int)
)
feature_df['near_cluster_size_exact_reps'] = (
    feature_df['near_cid']
    .map(cluster_size)
    .astype(int)
)
feature_df['keep_near'] = (
    feature_df['feature_index'] == feature_df['near_rep']
)

near_keep = np.sort(
    feature_df.loc[
        feature_df['keep_near'],
        'feature_index'
    ].to_numpy(dtype=int)
)

print("\nConservative near-dedup:")
print("  Retained :", f"{len(near_keep):,}")
print("  Removed  :", f"{n_docs-len(near_keep):,}",
      f"({100*(n_docs-len(near_keep))/n_docs:.2f}%)")
print("  Accepted near-duplicate pairs:", f"{len(accepted_pairs):,}")

# ============================================================
# INDEX INTEGRITY
# ============================================================

for name, idx in {
    'EXACT_DEDUP': exact_keep,
    'NEAR_DEDUP': near_keep,
}.items():
    if len(idx) == 0:
        raise ValueError(f"{name} is empty.")
    if idx.min() < 0 or idx.max() >= n_docs:
        raise IndexError(f"{name} contains invalid feature indices.")
    if len(np.unique(idx)) != len(idx):
        raise ValueError(f"{name} contains duplicate indices.")

print("\n✓ Sensitivity indices are valid feature-bundle row indices.")
print("✓ External dataset alignment is NOT required.")

# ============================================================
# EXPORT AUDIT CORPORA
# ============================================================

np.save(
    CORPUS_DIR/'full_keep_indices.npy',
    np.arange(n_docs, dtype=int)
)
np.save(
    CORPUS_DIR/'exact_keep_indices.npy',
    exact_keep
)
np.save(
    CORPUS_DIR/'near_keep_indices.npy',
    near_keep
)

feature_df.rename(
    columns={'_norm': 'normalized_text'}
).to_csv(
    CORPUS_DIR/'feature_bundle_duplicate_audit_map.csv',
    index=False
)

feature_df.loc[
    feature_df['keep_exact'],
    ['feature_index', 'raw_text', 'processed_text']
].to_csv(
    CORPUS_DIR/'feature_bundle_exact_dedup_corpus.csv',
    index=False
)

feature_df.loc[
    feature_df['keep_near'],
    ['feature_index', 'raw_text', 'processed_text']
].to_csv(
    CORPUS_DIR/'feature_bundle_near_dedup_corpus.csv',
    index=False
)

pair_rows = []
for a, bb, cosine_sim, jac, seq in accepted_pairs:
    pair_rows.append({
        'feature_index_a': int(rep.iloc[a].feature_index),
        'feature_index_b': int(rep.iloc[bb].feature_index),
        'tfidf_char_cosine': cosine_sim,
        'token_jaccard': jac,
        'sequence_ratio': seq,
        'text_a': texts[a],
        'text_b': texts[bb],
    })

pd.DataFrame(pair_rows).to_csv(
    CORPUS_DIR/'feature_bundle_near_duplicate_pairs.csv',
    index=False
)

audit_summary = pd.DataFrame([
    {
        'Condition': 'FULL',
        'Rows': n_docs,
        'Removed_vs_FULL': 0,
        'Removed_Percent': 0.0,
    },
    {
        'Condition': 'EXACT_DEDUP',
        'Rows': len(exact_keep),
        'Removed_vs_FULL': n_docs-len(exact_keep),
        'Removed_Percent': 100*(n_docs-len(exact_keep))/n_docs,
    },
    {
        'Condition': 'NEAR_DEDUP',
        'Rows': len(near_keep),
        'Removed_vs_FULL': n_docs-len(near_keep),
        'Removed_Percent': 100*(n_docs-len(near_keep))/n_docs,
    },
])

audit_summary.to_csv(
    TABLE_DIR/'duplicate_corpus_summary.csv',
    index=False
)

display(audit_summary)


## Locked HMRNet-Final


## 10-seed manuscript run — robust co-occurrence metrics

Eksperimen final memakai fixed HMRNet-Final configuration. C_NPMI dan UMass dihitung dengan satu implementasi document-level co-occurrence yang sama pada seluruh seed/kondisi; nilai Gensim asli tetap diekspor untuk audit.


In [ ]:

# ============================================================
# LOCKED HMRNet-FINAL + ROBUST 10-SEED SENSITIVITY EVALUATION
# ============================================================

ref = HMRNet.load(FINAL_CHECKPOINT, device=DEVICE)
ref_cfg = copy.deepcopy(ref.config)
ref_beta = ref.get_topic_word_distribution()

expected = {
    'n_topics': 20,
    'hidden_dim': 256,
    'attention_heads': 4,
    'learning_rate': 0.002,
    'dropout': 0.20,
    'weight_decay': 1e-5,
    'batch_size': 128,
    'epochs': 60,
    'patience': 16,
    'kl_weight': 0.10,
    'alignment_weight': 0.10,
    'diversity_weight': 0.07,
    'stability_weight': 0.00,
    'use_stability_loss': False,
    'branch_noise_std': 0.00,
    'use_cra': True,
    'fusion_mode': 'adaptive',
}

actual = asdict(ref_cfg)

mismatch = {
    k: {'actual': actual.get(k), 'expected': v}
    for k, v in expected.items()
    if actual.get(k) != v
}

if mismatch:
    raise ValueError(
        'Checkpoint config mismatch:\n'
        + json.dumps(mismatch, indent=2)
    )

print("✓ Locked HMRNet-Final configuration verified.")


# ============================================================
# GENERAL HELPERS
# ============================================================

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def metric_dict(metrics):
    if isinstance(metrics, dict):
        source = metrics
    elif isinstance(metrics, pd.Series):
        source = metrics.to_dict()
    elif isinstance(metrics, pd.DataFrame):
        source = {} if metrics.empty else metrics.iloc[0].to_dict()
    else:
        source = getattr(metrics, '__dict__', {})

    out = {}

    for k, v in source.items():
        if (
            isinstance(v, (int, float, np.integer, np.floating))
            and not isinstance(v, bool)
        ):
            out[k] = float(v)

    return out


def finite_float(value):
    try:
        value = float(value)
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan


def aligned_js(beta_a, beta_b):
    beta_a = np.asarray(beta_a, float)
    beta_b = np.asarray(beta_b, float)

    if beta_a.shape != beta_b.shape:
        raise ValueError(
            f"Beta shape mismatch: {beta_a.shape} vs {beta_b.shape}"
        )

    K = beta_a.shape[0]
    similarity = np.zeros((K, K), dtype=float)
    eps = 1e-12

    for i in range(K):
        p = np.clip(beta_a[i], eps, None)
        p = p / p.sum()

        for j in range(K):
            q = np.clip(beta_b[j], eps, None)
            q = q / q.sum()

            similarity[i, j] = (
                1.0
                - float(
                    jensenshannon(
                        p,
                        q,
                        base=2,
                    )
                )
            )

    row_ind, col_ind = linear_sum_assignment(-similarity)

    return float(
        similarity[
            row_ind,
            col_ind,
        ].mean()
    )


def fusion_summary(model, indices):
    try:
        weights = np.asarray(
            model.get_fusion_weights(
                bow=bow[indices],
                semantic_embeddings=sem[indices],
                document_embeddings=doc[indices],
            )
        )

        if weights.ndim == 2 and weights.shape[1] >= 3:
            avg = weights.mean(axis=0)

            return {
                'Fusion_Lexical': float(avg[0]),
                'Fusion_Semantic': float(avg[1]),
                'Fusion_Document': float(avg[2]),
            }

    except Exception as exc:
        print('Fusion weights skipped:', exc)

    return {}


# ============================================================
# ROBUST DOCUMENT-LEVEL COOCCURRENCE METRICS
# ============================================================

def prepare_eval_documents(processed_subset):
    """
    Convert evaluation documents to normalized strings while preserving
    the token sequence used by the project.

    Works whether processed_texts entries are strings or token lists.
    """
    docs = []

    for item in processed_subset:
        if isinstance(item, (list, tuple, np.ndarray)):
            text = " ".join(map(str, item))
        else:
            text = "" if item is None else str(item)

        text = re.sub(
            r"\s+",
            " ",
            text.strip(),
        )

        docs.append(text)

    return docs


def normalize_metric_term(term):
    """
    Conservative normalization for top words / n-grams.
    The aim is term matching, not re-preprocessing the corpus.
    """
    term = "" if term is None else str(term)
    term = re.sub(r"\s+", " ", term.strip())
    return term


def robust_npmi_umass(
    topics,
    processed_subset,
    top_n=10,
):
    """
    Deterministic document-level implementation for all sensitivity runs.

    C_NPMI for an unordered word pair (wi, wj):
        PMI = log(Pij / (Pi * Pj))
        NPMI = PMI / -log(Pij)

    Boundary handling:
        Pij = 0 -> NPMI = -1
        Pij = 1 -> NPMI = +1 only when both terms occur in every document

    Topic C_NPMI:
        arithmetic mean of all pairwise NPMI values in that topic.

    Corpus C_NPMI:
        arithmetic mean of topic C_NPMI values.

    UMass for ranked topic terms:
        log((D(w_m, w_l) + 1) / D(w_l)),  m > l

    Topic UMass:
        arithmetic mean over ordered rank pairs.

    Corpus UMass:
        arithmetic mean over topic UMass values.

    Multi-word topic terms are matched as exact n-grams using CountVectorizer
    with a fixed vocabulary. This is important because HMRNet topics contain
    terms such as "makan bergizi" and "bergizi gratis".
    """

    docs = prepare_eval_documents(processed_subset)

    N = len(docs)

    if N == 0:
        raise ValueError("Evaluation corpus is empty.")

    clean_topics = []
    all_terms = []

    for topic in topics:
        cleaned = []
        seen = set()

        for term in topic[:top_n]:
            term = normalize_metric_term(term)

            if term and term not in seen:
                cleaned.append(term)
                seen.add(term)

        clean_topics.append(cleaned)
        all_terms.extend(cleaned)

    terms = list(dict.fromkeys(all_terms))

    if not terms:
        raise ValueError("No valid topic terms found.")

    max_ngram = max(
        max(1, len(term.split()))
        for term in terms
    )

    vocabulary_map = {
        term: idx
        for idx, term in enumerate(terms)
    }

    vectorizer = CountVectorizer(
        vocabulary=vocabulary_map,
        lowercase=False,
        binary=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, max_ngram),
    )

    X = vectorizer.transform(docs)

    if X.nnz:
        X.data[:] = 1

    X = X.astype(np.int64)

    document_frequency = (
        np.asarray(
            X.sum(axis=0)
        )
        .ravel()
        .astype(np.int64)
    )

    cooccurrence = (
        X.T @ X
    ).toarray().astype(np.int64)

    npmi_topic_scores = []
    umass_topic_scores = []

    diagnostics = {
        'Metric_Total_Documents': int(N),
        'Metric_Unique_Terms': int(len(terms)),
        'Metric_Max_Ngram': int(max_ngram),
        'Metric_ZeroDF_Terms': 0,
        'Metric_NPMI_ZeroCooccurrence_Pairs': 0,
        'Metric_NPMI_Pair_Count': 0,
        'Metric_UMass_Pair_Count': 0,
        'Metric_UMass_Skipped_ZeroConditionDF': 0,
        'Metric_NPMI_Topics_Used': 0,
        'Metric_UMass_Topics_Used': 0,
    }

    zero_df_term_ids = set()

    for topic in clean_topics:
        cols = [
            vocabulary_map[t]
            for t in topic
            if t in vocabulary_map
        ]

        # -------------------------
        # C_NPMI
        # -------------------------
        topic_npmi = []

        for a in range(len(cols)):
            for b in range(a + 1, len(cols)):
                ia = cols[a]
                ib = cols[b]

                di = int(document_frequency[ia])
                dj = int(document_frequency[ib])
                dij = int(cooccurrence[ia, ib])

                if di == 0:
                    zero_df_term_ids.add(ia)

                if dj == 0:
                    zero_df_term_ids.add(ib)

                pi = di / N
                pj = dj / N
                pij = dij / N

                if dij == 0:
                    score = -1.0
                    diagnostics[
                        'Metric_NPMI_ZeroCooccurrence_Pairs'
                    ] += 1

                elif (
                    dij == N
                    and di == N
                    and dj == N
                ):
                    # Continuous limit for perfectly co-occurring terms.
                    score = 1.0

                else:
                    if pi <= 0 or pj <= 0 or pij <= 0:
                        # Guard should only be reached for unexpected data.
                        score = -1.0
                    else:
                        pmi = np.log(
                            pij / (pi * pj)
                        )

                        denom = -np.log(pij)

                        if denom <= 0:
                            score = 1.0
                        else:
                            score = pmi / denom

                        score = float(
                            np.clip(
                                score,
                                -1.0,
                                1.0,
                            )
                        )

                topic_npmi.append(float(score))

                diagnostics[
                    'Metric_NPMI_Pair_Count'
                ] += 1

        if topic_npmi:
            npmi_topic_scores.append(
                float(np.mean(topic_npmi))
            )

        # -------------------------
        # UMass
        # -------------------------
        topic_umass = []

        # Respect top-word rank:
        # m is lower-ranked term, l is earlier/higher-ranked term.
        for m in range(1, len(cols)):
            for l in range(m):
                im = cols[m]
                il = cols[l]

                d_l = int(
                    document_frequency[il]
                )

                d_ml = int(
                    cooccurrence[im, il]
                )

                if d_l <= 0:
                    zero_df_term_ids.add(il)

                    diagnostics[
                        'Metric_UMass_Skipped_ZeroConditionDF'
                    ] += 1

                    continue

                score = np.log(
                    (d_ml + 1.0)
                    / d_l
                )

                topic_umass.append(
                    float(score)
                )

                diagnostics[
                    'Metric_UMass_Pair_Count'
                ] += 1

        if topic_umass:
            umass_topic_scores.append(
                float(np.mean(topic_umass))
            )

    diagnostics[
        'Metric_ZeroDF_Terms'
    ] = int(
        len(zero_df_term_ids)
    )

    diagnostics[
        'Metric_NPMI_Topics_Used'
    ] = int(
        len(npmi_topic_scores)
    )

    diagnostics[
        'Metric_UMass_Topics_Used'
    ] = int(
        len(umass_topic_scores)
    )

    if len(npmi_topic_scores) != len(clean_topics):
        raise ValueError(
            "Robust NPMI did not produce one score per topic: "
            f"{len(npmi_topic_scores)}/{len(clean_topics)}"
        )

    if len(umass_topic_scores) != len(clean_topics):
        raise ValueError(
            "Robust UMass did not produce one score per topic: "
            f"{len(umass_topic_scores)}/{len(clean_topics)}"
        )

    c_npmi = float(
        np.mean(
            npmi_topic_scores
        )
    )

    umass = float(
        np.mean(
            umass_topic_scores
        )
    )

    if not np.isfinite(c_npmi):
        raise ValueError(
            f"Robust C_NPMI is not finite: {c_npmi}"
        )

    if not np.isfinite(umass):
        raise ValueError(
            f"Robust UMass is not finite: {umass}"
        )

    return c_npmi, umass, diagnostics


def attach_robust_coherence(
    row,
    topics,
    evaluation_texts,
):
    """
    Keep original project/Gensim values for audit, but use the robust
    implementation as the sensitivity-study C_NPMI and UMass for EVERY run.
    """

    row = dict(row)

    row[
        'C_NPMI_Gensim_Audit'
    ] = finite_float(
        row.get(
            'C_NPMI',
            np.nan,
        )
    )

    row[
        'UMass_Gensim_Audit'
    ] = finite_float(
        row.get(
            'UMass',
            np.nan,
        )
    )

    robust_npmi, robust_umass, diag = (
        robust_npmi_umass(
            topics=topics,
            processed_subset=
                evaluation_texts,
            top_n=
                TOP_N_WORDS,
        )
    )

    row[
        'C_NPMI_Robust'
    ] = robust_npmi

    row[
        'UMass_Robust'
    ] = robust_umass

    # The manuscript-facing sensitivity metrics use one definition
    # consistently for all conditions and all seeds.
    row[
        'C_NPMI'
    ] = robust_npmi

    row[
        'UMass'
    ] = robust_umass

    row.update(diag)

    return row


# ============================================================
# CONDITIONS + V7 OUTPUT DIRECTORIES
# ============================================================

conditions = {
    'FULL':
        np.arange(
            n_docs,
            dtype=int,
        ),
    'EXACT_DEDUP':
        exact_keep,
    'NEAR_DEDUP':
        near_keep,
}

V7_RUN_DIR = (
    OUTPUT_DIR
    / 'runs_v7_10seed'
)

V7_TABLE_DIR = (
    OUTPUT_DIR
    / 'tables_v7_10seed'
)

V7_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

V7_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

rows = []

betas = {
    name: {}
    for name in conditions
}

pairwise_frames = []


# ============================================================
# 10-SEED TRAINING / CACHE REUSE
# ============================================================

for condition_name, indices in (
    conditions.items()
):

    evaluation_texts = [
        processed[i]
        for i in indices
    ]

    print(
        "\n"
        + "=" * 100
    )

    print(
        "CONDITION:",
        condition_name,
        "| N =",
        len(indices),
        "| seeds =",
        SEEDS,
    )

    print(
        "=" * 100
    )

    for seed in SEEDS:

        run_id = (
            f'{condition_name.lower()}'
            f'_seed_{seed}'
        )

        # Quick-run cache from v4.
        quick_cache = (
            RUN_DIR
            / f'{run_id}.joblib'
        )

        # V7 final cache.
        v7_cache = (
            V7_RUN_DIR
            / f'{run_id}.joblib'
        )

        model_path = (
            MODEL_DIR
            / (
                'HMRNet-Final-'
                f'{run_id}.pt'
            )
        )

        # ----------------------------------------------------
        # 1. Reuse completed V7 run
        # ----------------------------------------------------
        if (
            v7_cache.exists()
            and not FORCE_RETRAIN
        ):
            payload = joblib.load(
                v7_cache
            )

            row = dict(
                payload['row']
            )

            topics = payload[
                'topics'
            ]

            beta = payload[
                'beta'
            ]

            print(
                '[V7 CACHE]',
                run_id,
            )

        # ----------------------------------------------------
        # 2. Reuse quick seed 1-3 checkpoint/cache
        # ----------------------------------------------------
        elif (
            quick_cache.exists()
            and not FORCE_RETRAIN
        ):
            old_payload = joblib.load(
                quick_cache
            )

            required = [
                'row',
                'topics',
                'beta',
            ]

            missing = [
                key
                for key in required
                if key not in old_payload
            ]

            if missing:
                raise KeyError(
                    f"{quick_cache} missing keys: "
                    f"{missing}"
                )

            row = dict(
                old_payload[
                    'row'
                ]
            )

            topics = old_payload[
                'topics'
            ]

            beta = old_payload[
                'beta'
            ]

            row = (
                attach_robust_coherence(
                    row=row,
                    topics=topics,
                    evaluation_texts=
                        evaluation_texts,
                )
            )

            payload = {
                'row':
                    row,
                'topics':
                    topics,
                'beta':
                    beta,
                'indices':
                    indices,
                'config':
                    old_payload.get(
                        'config',
                        asdict(
                            ref_cfg
                        ),
                    ),
                'source':
                    'quick_cache_reused'
                    '_with_v7_robust_metrics',
            }

            joblib.dump(
                payload,
                v7_cache,
            )

            print(
                '[QUICK CACHE + '
                'V7 ROBUST METRICS]',
                run_id,
            )

        # ----------------------------------------------------
        # 3. Train missing seed, normally 4-10
        # ----------------------------------------------------
        else:
            print(
                '[TRAIN]',
                run_id,
            )

            seed_all(seed)

            cfg = copy.deepcopy(
                ref_cfg
            )

            cfg.random_state = int(
                seed
            )

            cfg.device = DEVICE
            cfg.epochs = 60
            cfg.patience = 16

            model = HMRNet(cfg)

            model.fit(
                bow=bow[indices],
                semantic_embeddings=
                    sem[indices],
                document_embeddings=
                    doc[indices],
                vocabulary=vocab,
                verbose=True,
            )

            model.save(
                model_path
            )

            topics = (
                model.get_topics(
                    top_n=
                        TOP_N_WORDS
                )
            )

            # Keep original project evaluator for C_v and audit.
            project_metrics = (
                evaluate_topics(
                    model_name=
                        run_id,
                    topics=
                        topics,
                    processed_texts=
                        evaluation_texts,
                    top_n=
                        TOP_N_WORDS,
                )
            )

            base_row = (
                metric_dict(
                    project_metrics
                )
            )

            beta = (
                model
                .get_topic_word_distribution()
            )

            row = {
                'Condition':
                    condition_name,
                'Seed':
                    int(seed),
                'N_Documents':
                    int(
                        len(indices)
                    ),
                **base_row,
                'Aligned_JS_Similarity_to_Final_Checkpoint':
                    aligned_js(
                        ref_beta,
                        beta,
                    ),
                **fusion_summary(
                    model,
                    indices,
                ),
                'Model_Path':
                    str(
                        model_path
                    ),
            }

            row = (
                attach_robust_coherence(
                    row=row,
                    topics=topics,
                    evaluation_texts=
                        evaluation_texts,
                )
            )

            payload = {
                'row':
                    row,
                'topics':
                    topics,
                'beta':
                    beta,
                'indices':
                    indices,
                'config':
                    asdict(cfg),
                'source':
                    'new_10seed_training'
                    '_with_v7_robust_metrics',
            }

            joblib.dump(
                payload,
                v7_cache,
            )

            del model
            gc.collect()

            if (
                torch.cuda
                .is_available()
            ):
                torch.cuda.empty_cache()

        # Mandatory finite guard for V7 manuscript metrics.
        for metric in [
            'C_NPMI',
            'UMass',
        ]:
            value = finite_float(
                row.get(
                    metric,
                    np.nan,
                )
            )

            if not np.isfinite(
                value
            ):
                raise ValueError(
                    f"{run_id}: "
                    f"{metric} remains "
                    f"non-finite."
                )

        rows.append(
            row
        )

        betas[
            condition_name
        ][seed] = beta

    # --------------------------------------------------------
    # 10-seed stability
    # --------------------------------------------------------
    pairwise, stability = (
        evaluate_stability_runs(
            beta_by_run=
                betas[
                    condition_name
                ],
            metric='js',
            confidence=0.95,
        )
    )

    pairwise[
        'Condition'
    ] = condition_name

    pairwise_frames.append(
        pairwise
    )

    print(
        condition_name,
        'Mean_TS =',
        stability.get(
            'Mean_TS'
        ),
    )


# ============================================================
# RUN-LEVEL TABLE
# ============================================================

runs = (
    pd.DataFrame(
        rows
    )
    .sort_values(
        [
            'Condition',
            'Seed',
        ]
    )
    .reset_index(
        drop=True
    )
)

runs.to_csv(
    V7_TABLE_DIR
    / 'duplicate_sensitivity_10seed_all_runs.csv',
    index=False,
)


# ============================================================
# COOCCURRENCE / GENSIM AUDIT
# ============================================================

audit_cols = [
    'Condition',
    'Seed',
    'C_v',
    'C_NPMI_Gensim_Audit',
    'C_NPMI_Robust',
    'UMass_Gensim_Audit',
    'UMass_Robust',
    'Metric_Total_Documents',
    'Metric_Unique_Terms',
    'Metric_Max_Ngram',
    'Metric_ZeroDF_Terms',
    'Metric_NPMI_ZeroCooccurrence_Pairs',
    'Metric_NPMI_Pair_Count',
    'Metric_UMass_Pair_Count',
    'Metric_UMass_Skipped_ZeroConditionDF',
    'Metric_NPMI_Topics_Used',
    'Metric_UMass_Topics_Used',
]

runs[
    [
        c
        for c in audit_cols
        if c in runs.columns
    ]
].to_csv(
    V7_TABLE_DIR
    / 'coherence_metric_audit_10seed.csv',
    index=False,
)


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_metrics = [
    'C_v',
    'C_NPMI',
    'UMass',
    'Topic_Diversity',
    'Topic_Redundancy',
    'Topic_Uniqueness',
    'Aligned_JS_Similarity_to_Final_Checkpoint',
    'Fusion_Lexical',
    'Fusion_Semantic',
    'Fusion_Document',
]

summary_rows = []

for condition_name, group in (
    runs.groupby(
        'Condition',
        sort=False,
    )
):

    out = {
        'Condition':
            condition_name,
        'N_Documents':
            int(
                group[
                    'N_Documents'
                ].iloc[0]
            ),
        'N_Seeds':
            int(
                group[
                    'Seed'
                ].nunique()
            ),
    }

    for metric in (
        summary_metrics
    ):
        if metric not in (
            group.columns
        ):
            continue

        values = (
            pd.to_numeric(
                group[metric],
                errors='coerce',
            )
        )

        finite_values = values[
            np.isfinite(
                values
            )
        ]

        if len(finite_values) != 10:
            raise ValueError(
                f"{condition_name} {metric}: "
                f"only {len(finite_values)}/10 "
                f"finite values."
            )

        out[
            f'{metric}_Mean'
        ] = float(
            finite_values.mean()
        )

        out[
            f'{metric}_SD'
        ] = float(
            finite_values.std(
                ddof=1
            )
        )

    _, stability = (
        evaluate_stability_runs(
            beta_by_run=
                betas[
                    condition_name
                ],
            metric='js',
            confidence=0.95,
        )
    )

    out.update(
        stability
    )

    summary_rows.append(
        out
    )

summary = pd.DataFrame(
    summary_rows
)

full = summary.loc[
    summary[
        'Condition'
    ].eq(
        'FULL'
    )
].iloc[0]

for metric in [
    'C_v_Mean',
    'C_NPMI_Mean',
    'UMass_Mean',
    'Topic_Diversity_Mean',
    'Topic_Redundancy_Mean',
    'Mean_TS',
]:
    if metric in (
        summary.columns
    ):
        summary[
            f'Delta_{metric}'
            f'_vs_FULL'
        ] = (
            summary[metric]
            - float(
                full[metric]
            )
        )

summary.to_csv(
    V7_TABLE_DIR
    / 'duplicate_sensitivity_10seed_summary.csv',
    index=False,
)


# ============================================================
# STABILITY + TOPIC WORDS
# ============================================================

pd.concat(
    pairwise_frames,
    ignore_index=True,
).to_csv(
    V7_TABLE_DIR
    / 'duplicate_sensitivity_10seed_stability_pairwise.csv',
    index=False,
)

topic_rows = []

for condition_name in (
    conditions
):
    for seed in SEEDS:
        cache_path = (
            V7_RUN_DIR
            / (
                f'{condition_name.lower()}'
                f'_seed_{seed}.joblib'
            )
        )

        payload = joblib.load(
            cache_path
        )

        for topic_id, words in (
            enumerate(
                payload[
                    'topics'
                ]
            )
        ):
            topic_rows.append(
                {
                    'Condition':
                        condition_name,
                    'Seed':
                        int(seed),
                    'Topic':
                        int(topic_id),
                    'Top_Words':
                        ', '.join(
                            words
                        ),
                }
            )

pd.DataFrame(
    topic_rows
).to_csv(
    V7_TABLE_DIR
    / 'duplicate_sensitivity_10seed_topic_words.csv',
    index=False,
)


# ============================================================
# MANIFEST
# ============================================================

manifest = {
    'mode':
        'manuscript',
    'seeds':
        list(
            range(
                1,
                11,
            )
        ),
    'index_source':
        'fixed_feature_bundle_raw_texts',
    'external_dataset_alignment_required':
        False,
    'coherence_policy': {
        'C_v':
            'project evaluate_topics / Gensim',
        'C_NPMI_final':
            'deterministic document-level binary cooccurrence; same formula for all 30 sensitivity runs',
        'UMass_final':
            'deterministic document-level binary cooccurrence; same formula for all 30 sensitivity runs',
        'Gensim_C_NPMI_UMass':
            'retained as audit columns only',
        'phrase_handling':
            'exact n-gram matching via fixed-vocabulary CountVectorizer',
        'reason':
            'Gensim aggregate C_NPMI/UMass produced non-finite values for some HMRNet runs; one robust definition is therefore applied consistently across all sensitivity conditions/seeds.',
    },
    'duplicate_thresholds': {
        'tfidf_cosine':
            TFIDF_COSINE_CANDIDATE,
        'token_jaccard':
            TOKEN_JACCARD_THRESHOLD,
        'sequence_ratio':
            SEQUENCE_RATIO_THRESHOLD,
        'k_neighbors':
            K_NEIGHBORS,
    },
    'locked_config':
        expected,
}

(
    OUTPUT_DIR
    / 'duplicate_sensitivity_10seed_manifest_v7.json'
).write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)


# ============================================================
# FINAL VALIDATION
# ============================================================

expected_runs = (
    len(conditions)
    * len(SEEDS)
)

if len(runs) != expected_runs:
    raise ValueError(
        f"Expected {expected_runs} runs, "
        f"but found {len(runs)}."
    )

for metric in [
    'C_NPMI',
    'UMass',
]:
    vals = pd.to_numeric(
        runs[metric],
        errors='coerce',
    )

    if (
        ~np.isfinite(
            vals
        )
    ).any():
        raise ValueError(
            f"Non-finite values remain "
            f"in final {metric}."
        )

print(
    "\n"
    + "=" * 104
)

print(
    "FINAL 10-SEED DUPLICATE "
    "SENSITIVITY SUMMARY — V7"
)

print(
    "=" * 104
)

print(
    summary
    .round(6)
    .to_string(
        index=False
    )
)

print(
    "\n✓ 30 runs verified "
    "(3 conditions × 10 seeds)."
)

print(
    "✓ Final C_NPMI has no NaN/Inf."
)

print(
    "✓ Final UMass has no NaN/Inf."
)

print(
    "✓ One robust co-occurrence definition "
    "used for all runs."
)

print(
    "✓ Original Gensim C_NPMI/UMass "
    "retained in audit table."
)

print(
    "\nMain output:"
)

print(
    V7_TABLE_DIR
    / 'duplicate_sensitivity_10seed_summary.csv'
)
